# Narrating long Indic text with Sarvam streaming text to speech

Turning a long Hindi passage into audio means cutting it into chunks the speech API will
accept. The usual way to cut sentences - look for a full stop followed by a space - finds
nothing in Hindi, because a Hindi sentence ends with a danda, `।`, not a full stop. The
whole passage goes to the API as one oversized lump.

Pipeline:

1. Show the failure on a 3,240-character Hindi passage, using the splitter that ships in
   `examples/tts/book__summary_narrator.ipynb` today. No API key needed.
2. Cut the same passage with `indic_splitter.split_for_tts`, which knows about the danda and
   never breaks a grapheme cluster. No API key needed.
3. Stream each chunk through `text_to_speech.convert_stream` and write the bytes to a file in
   `outputs/` as they arrive. This step needs an API key.

**Every cell in this notebook ships with empty output. It has not been run against the live
Sarvam API - there is no key on the machine that produced it. See the README.** Run the cells
yourself and the output you see will be your own.

In [ ]:
%pip install -r requirements.txt

## Step 1: the failure, on real Hindi

`examples/tts/book__summary_narrator.ipynb` cuts sentences like this, in cell 15:

```python
sentences = text.replace('\n', ' ').split('. ')
```

Its budget, `MAX_CHUNK_LENGTH`, is 500 characters, set in cell 9 of the same notebook.

The cell below is that function copied verbatim, run over a 3,240-character Hindi passage
whose sentences all end in dandas. It is plain string work: no API key, no network. The
passage is ordinary agricultural-advisory prose written for this recipe, so there is no
licensing question about it.

In [ ]:
from __future__ import annotations

MAX_CHUNK_LENGTH = 500  # examples/tts/book__summary_narrator.ipynb, cell 9


def split_text_into_chunks(text, max_length=MAX_CHUNK_LENGTH):
    """The splitter that ships in examples/tts/book__summary_narrator.ipynb, cell 15."""
    sentences = text.replace("\n", " ").split(". ")
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if sentence != sentences[-1]:
            sentence += "."

        if len(current_chunk) + len(sentence) + 1 > max_length:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
        else:
            current_chunk += sentence + " "

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


HINDI_PASSAGE = (
    "मानसून भारत की खेती की रीढ़ है और हर साल जून के पहले सप्ताह में केरल के तट से इसकी शुरुआत होती है। "
    "किसान इस बारिश का इंतजार महीनों पहले से करते हैं क्योंकि खरीफ की पूरी फसल इसी पानी पर टिकी होती है। "
    "मौसम विभाग हर सुबह और हर शाम अपने पूर्वानुमान को अद्यतन करता है ताकि गाँव तक सही जानकारी पहुँच सके। "
    "अगर बारिश तय समय से एक पखवाड़े देर से आती है तो धान की रोपाई पिछड़ जाती है और पैदावार घट जाती है। "
    "इसी वजह से राज्य सरकारें बुवाई का कार्यक्रम मौसम की चेतावनी के साथ जोड़कर तैयार करती हैं। "
    "छोटे किसानों के पास सिंचाई का अपना साधन नहीं होता इसलिए वे पूरी तरह आसमान पर निर्भर रहते हैं। "
    "कृषि विज्ञान केंद्र के कर्मचारी हर हफ्ते खेतों में जाकर मिट्टी की नमी की जाँच करते हैं। "
    "नमी कम मिलने पर वे कम पानी माँगने वाली फसलों की सलाह देते हैं जैसे बाजरा ज्वार और मूँग। "
    "बीज की गुणवत्ता भी उतनी ही अहम है जितना समय पर हुई बारिश और सही मात्रा में डाला गया खाद। "
    "सरकारी केंद्रों से प्रमाणित बीज लेने पर अंकुरण की दर बहुत बेहतर रहती है यह बात अनुभव से सिद्ध है। "
    "कीट लगने की शुरुआती पहचान कर लेने से दवा का खर्च आधा रह जाता है और फसल भी सुरक्षित बच जाती है। "
    "इसलिए विशेषज्ञ सलाह देते हैं कि हर तीसरे दिन खेत का एक चक्कर जरूर लगाना चाहिए। "
    "कटाई के बाद अनाज को अच्छी तरह सुखाना उतना ही जरूरी है वरना भंडारण में फफूँद लग जाती है। "
    "मंडी तक पहुँचने से पहले तौल और नमी की जाँच करा लेना किसान के हित में रहता है। "
    "बहुत से किसान अब मोबाइल पर ही मंडी के भाव देख लेते हैं और उसी हिसाब से बेचने का दिन तय करते हैं। "
    "यह बदलाव पिछले कुछ वर्षों में तेजी से आया है और इसका सीधा लाभ छोटे किसानों को मिला है। "
    "फिर भी सबसे बड़ी चुनौती यही है कि सही जानकारी उस भाषा में मिले जिसे किसान आसानी से समझ सके। "
    "इसीलिए मौसम और खेती की सलाह को स्थानीय भाषा में सुनाकर पहुँचाना सबसे कारगर तरीका माना जाता है। "
    "सुनकर समझने वाली सलाह पढ़ने वाली सलाह से कहीं ज्यादा लोगों तक पहुँचती है यह सर्वेक्षण में पाया गया। "
    "गाँव के चौपाल पर एक साथ बैठकर सुनी गई बात कई घरों तक अपने आप फैल जाती है। "
    "मानसून भारत की खेती की रीढ़ है और हर साल जून के पहले सप्ताह में केरल के तट से इसकी शुरुआत होती है। "
    "किसान इस बारिश का इंतजार महीनों पहले से करते हैं क्योंकि खरीफ की पूरी फसल इसी पानी पर टिकी होती है। "
    "मौसम विभाग हर सुबह और हर शाम अपने पूर्वानुमान को अद्यतन करता है ताकि गाँव तक सही जानकारी पहुँच सके। "
    "अगर बारिश तय समय से एक पखवाड़े देर से आती है तो धान की रोपाई पिछड़ जाती है और पैदावार घट जाती है। "
    "इसी वजह से राज्य सरकारें बुवाई का कार्यक्रम मौसम की चेतावनी के साथ जोड़कर तैयार करती हैं। "
    "छोटे किसानों के पास सिंचाई का अपना साधन नहीं होता इसलिए वे पूरी तरह आसमान पर निर्भर रहते हैं। "
    "कृषि विज्ञान केंद्र के कर्मचारी हर हफ्ते खेतों में जाकर मिट्टी की नमी की जाँच करते हैं। "
    "नमी कम मिलने पर वे कम पानी माँगने वाली फसलों की सलाह देते हैं जैसे बाजरा ज्वार और मूँग। "
    "बीज की गुणवत्ता भी उतनी ही अहम है जितना समय पर हुई बारिश और सही मात्रा में डाला गया खाद। "
    "सरकारी केंद्रों से प्रमाणित बीज लेने पर अंकुरण की दर बहुत बेहतर रहती है यह बात अनुभव से सिद्ध है। "
    "कीट लगने की शुरुआती पहचान कर लेने से दवा का खर्च आधा रह जाता है और फसल भी सुरक्षित बच जाती है। "
    "इसलिए विशेषज्ञ सलाह देते हैं कि हर तीसरे दिन खेत का एक चक्कर जरूर लगाना चाहिए। "
    "कटाई के बाद अनाज को अच्छी तरह सुखाना उतना ही जरूरी है वरना भंडारण में फफूँद लग जाती है। "
    "मंडी तक पहुँचने से पहले तौल और नमी की जाँच करा लेना किसान के हित में रहता है। "
    "बहुत से किसान अब मोबाइल पर ही मंडी के भाव देख लेते हैं और उसी हिसाब से बेचने का दिन तय करते हैं। "
    "किसान मौसम बारिश फसल खेत गाँव।"
)

ENGLISH_PASSAGE = (
    "The monsoon is the backbone of Indian farming and it reaches the Kerala coast in the "
    "first week of June. Farmers wait months for this rain because the whole kharif crop "
    "rests on it. The weather office updates its forecast every morning and every evening. "
    "If the rain is a fortnight late the paddy transplanting slips and the yield falls. "
    "Small farmers have no irrigation of their own and depend entirely on the sky. "
    "State governments therefore tie the sowing calendar to the weather warnings. "
    "Extension staff walk the fields every week and test the soil for moisture. "
    "Where the reading is low they suggest crops that ask for less water, such as millet. "
    "The village noticeboard carries the grain rate every morning so nobody has to guess. "
    "After the rain stops the crop still needs looking after for several more weeks. "
    "Saving water in the first month leaves more of it for the month that matters most."
)


def report(label, text, chunks, budget):
    over = sum(1 for c in chunks if len(c) > budget)
    print(
        f"{label:<8} chars={len(text):<5} '. ' found={text.count('. '):<3} "
        f"danda found={text.count(chr(0x0964)):<3} "
        f"chunks={len(chunks):<3} largest={max(len(c) for c in chunks):<5} over budget={over}"
    )


for budget in (500, 2500):
    print(f"budget {budget}")
    report("Hindi", HINDI_PASSAGE, split_text_into_chunks(HINDI_PASSAGE, budget), budget)
    report("English", ENGLISH_PASSAGE, split_text_into_chunks(ENGLISH_PASSAGE, budget), budget)
    print()

## Step 2: the same passage, cut on dandas

`indic_splitter.py` sits next to this notebook. It is standard library only - it imports
nothing outside `unicodedata`, never touches the network, and never imports `sarvamai`, so the
test suite can exercise it without a key. `tests/test_indic_splitter.py` covers it.

`split_for_tts(text, max_chars=2500)` cuts at a sentence terminator when one is within the
budget, at a word boundary when it is not, and never inside a grapheme cluster. Three details
that are easy to get wrong and are all handled there:

- `unicodedata.combining()` returns **0** for Indic vowel signs, so the obvious "do not split
  before a combining mark" guard does nothing at all. The check has to be
  `unicodedata.category(ch) in ("Mn", "Mc")`.
- Zero-width joiner and non-joiner are category `Cf`, which the `Mn`/`Mc` check does not
  cover, so they get a rule of their own.
- Malayalam has three viramas, not one. The virama set is derived with
  `unicodedata.combining(ch) == 9` rather than hardcoded.

Nothing is stripped, so `"".join(chunks) == text` exactly. Trim at the call site if you want
tidy text for display.

In [ ]:
from indic_splitter import split_for_tts

BUDGET = 2500  # convert_stream allows 3500; 2500 also fits convert with bulbul:v3

chunks = split_for_tts(HINDI_PASSAGE, BUDGET)

print("chunks           :", len(chunks))
print("largest chunk    :", max(len(c) for c in chunks))
print("chunks over budget:", sum(1 for c in chunks if len(c) > BUDGET))
print("nothing lost     :", "".join(chunks) == HINDI_PASSAGE)
print()

for index, chunk in enumerate(chunks, start=1):
    tail = chunk.strip()[-45:]
    print(f"{index:>3}  {len(chunk):>5} chars  ends: ...{tail}")

## Setup

Copy `.env.example` to `.env` and paste your key into it. Get a key from the
[Sarvam dashboard](https://dashboard.sarvam.ai/).

The key has to be passed to `SarvamAI` explicitly. `SarvamAI.__init__` reads the environment
in a **default argument**, which Python evaluates once when the module is imported, so this
sequence fails even though it looks right:

```python
from sarvamai import SarvamAI   # the default is frozen to None here
load_dotenv()                   # too late
SarvamAI()                      # ApiError
```

`SarvamAI(api_subscription_key=os.environ["SARVAM_API_KEY"])` is the form that works.

In [ ]:
import os
import shutil
from pathlib import Path

from dotenv import load_dotenv
from sarvamai import SarvamAI

load_dotenv()

if not os.environ.get("SARVAM_API_KEY"):
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment, or copy .env.example to .env and "
        "put your key in it, before running this cell."
    )

client = SarvamAI(api_subscription_key=os.environ["SARVAM_API_KEY"])

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("client ready, writing to", OUTPUT_DIR.resolve())

## The streaming layer

`convert_stream` returns `typing.Iterator[bytes]` - raw audio, not a response object. Two
things follow from that, and both are easy to get wrong by copying from the older narrator
notebook:

- **`sarvamai.play.save()` cannot consume it.** `save()` expects a `TextToSpeechResponse` and
  reads `audio.audios` off it, so passing this iterator raises
  `AttributeError: 'generator' object has no attribute 'audios'`. Write the bytes to an open
  file handle instead.
- **Consume it while it is open.** `narrate_chunks` writes each block the moment it arrives
  rather than collecting the whole stream in memory first. That is the entire reason to use
  the streaming endpoint.

Two more, from the API surface rather than from the type:

- The parameter is `language_code`, not `target_language_code`.
- `model` defaults to `bulbul:v2`. Pass `bulbul:v3` on every call.

Defining these functions needs no API key. Calling them does.

In [ ]:
MODEL = "bulbul:v3"
SPEAKER = "shubh"   # the bulbul:v3 default; the enum holds 44 voices
CODEC = "mp3"


def narrate_chunks(
    client: SarvamAI,
    chunks: list[str],
    language_code: str,
    out_dir: Path,
    speaker: str = SPEAKER,
    codec: str = CODEC,
) -> list[Path]:
    """Stream every chunk to its own audio file and return the paths written."""
    paths: list[Path] = []
    for index, chunk in enumerate(chunks, start=1):
        path = out_dir / f"chunk_{index:02d}.{codec}"
        with path.open("wb") as sink:
            for block in client.text_to_speech.convert_stream(
                text=chunk,
                language_code=language_code,
                speaker=speaker,
                model=MODEL,
                output_audio_codec=codec,
            ):
                sink.write(block)
        paths.append(path)
    return paths


def join_audio(paths: list[Path], destination: Path) -> Path:
    """Append the per-chunk files end to end into one file."""
    with destination.open("wb") as sink:
        for path in paths:
            with path.open("rb") as source:
                shutil.copyfileobj(source, sink)
    return destination

## Step 3: write the audio

**This cell was not run.** There is no Sarvam API key on the machine that produced this
notebook, so it ships with empty output rather than with numbers copied from somewhere else.
Run it with your own key and the file sizes below will be real ones.

It writes one file per chunk plus `narration.mp3`, the chunk files appended end to end. That
concatenation is a byte append, not an audio splice - MP3 frames survive it and most players
handle the result, but it is not a substitute for proper audio editing.

Everything lands in `outputs/`, which is gitignored, so no audio is committed.

In [ ]:
LANGUAGE_CODE = "hi-IN"   # one of the 11 TTS codes; see the note below on Odia

chunk_paths = narrate_chunks(client, chunks, LANGUAGE_CODE, OUTPUT_DIR)
narration = join_audio(chunk_paths, OUTPUT_DIR / "narration.mp3")

for path in chunk_paths:
    print(f"{path.name:>16}  {path.stat().st_size:>9,} bytes")
print(f"{narration.name:>16}  {narration.stat().st_size:>9,} bytes")

## Changing the language

`language_code` takes one of eleven values, and only these eleven:

`bn-IN` `en-IN` `gu-IN` `hi-IN` `kn-IN` `ml-IN` `mr-IN` `od-IN` `pa-IN` `ta-IN` `te-IN`

Odia is `od-IN`. `or-IN` is valid on the dubbing and realtime streaming endpoints but is not
in the text-to-speech list, so it comes back as a 400. Speech-to-text covers many more
languages than this; that does not mean speech synthesis does. Assamese is the usual
assumption and it is not on the list.

Nothing is checked on your machine. Every enumerated value in the SDK is typed
`Union[Literal[...], Any]`, so a wrong language code, speaker or model is caught by neither
the runtime nor a type checker - it comes back as a 400 or 422 from the server.

Swap `HINDI_PASSAGE` for your own text and `LANGUAGE_CODE` for its language. The splitter
handles all eleven scripts; `tests/test_indic_splitter.py` has a passage for each one.